# DNS Shield — SOC Demo and Evidence Analysis

This notebook is an **offline analysis and presentation companion**. It does not make DNS decisions. Once the stack is running, it calls the API Gateway and turns real event data into charts and evidence.


## 1. Connection configuration and health check

In [ ]:
import os, requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

GATEWAY_URL = os.getenv('DNS_SHIELD_GATEWAY_URL', 'http://localhost:8080').rstrip('/')
API_KEY = os.getenv('DNS_SHIELD_API_KEY', '')
HEADERS = {'X-DNS-Shield-Key': API_KEY} if API_KEY else {}
print(f'Gateway: {GATEWAY_URL}')

def api_get(path, **params):
    try:
        response = requests.get(f'{GATEWAY_URL}{path}', headers=HEADERS, params=params, timeout=5)
        if response.status_code != 200:
            return {}
        return response.json()
    except requests.exceptions.RequestException:
        return {}

def api_post(path, payload):
    try:
        response = requests.post(f'{GATEWAY_URL}{path}', headers=HEADERS, json=payload, timeout=5)
        if response.status_code != 200:
            return {'domain': payload.get('domain', ''), 'verdict': 'UNAVAILABLE', 'domain_risk': 0, 'device_risk': 0, 'confidence': 'OFFLINE', 'latency_ms': 0, 'reasons': [f'Status {response.status_code}']}
        return response.json()
    except requests.exceptions.RequestException:
        return {'domain': payload.get('domain', ''), 'verdict': 'UNAVAILABLE', 'domain_risk': 0, 'device_risk': 0, 'confidence': 'OFFLINE', 'latency_ms': 0, 'reasons': ['Gateway offline']}

health = api_get('/health')
print('Gateway Status:', health.get('status', 'Offline'))

## 2. Run controlled pipeline demonstrations

In [ ]:
DEMO_CASES = [
    ('known-bad', 'c2.bad-demo.example'),
    ('benign', 'isro.gov.in'),
    ('dga-style', 'xq9m2kz7v4na.com'),
    ('typosquat', 'gooogle.com'),
]

results = {}
for name, domain in DEMO_CASES:
    results[name] = api_post('/v1/query', {
        'domain': domain,
        'client_ip': '172.28.0.120',
        'source': f'notebook:{name}',
    })

summary_df = pd.DataFrame([
    {'case': name, 'domain': result.get('domain', ''), 'verdict': result.get('verdict', 'OFFLINE'),
     'domain_risk': result.get('domain_risk', 0), 'device_risk': result.get('device_risk', 0),
     'confidence': result.get('confidence', 'NONE'), 'latency_ms': result.get('latency_ms', 0)}
    for name, result in results.items()
])
display(summary_df)

## 3. Explainable pipeline stage contributions (XAI)

In [ ]:
selected_case = 'known-bad'
selected = results.get(selected_case, {})
pipeline = pd.DataFrame(selected.get('pipeline', []))
if not pipeline.empty and {'stage', 'status', 'contribution', 'reason'}.issubset(pipeline.columns):
    display(pipeline[['stage', 'status', 'contribution', 'reason']])
    fig = px.bar(pipeline, x='stage', y='contribution', color='status',
                 hover_data=['reason'], title=f"{selected.get('domain', '')} — {selected.get('verdict', '')} XAI stage contributions")
    fig.update_layout(yaxis_title='Risk contribution', xaxis_title='Pipeline stage')
    fig.show()
else:
    print('Pipeline stage data unavailable (gateway offline or query not run).')

print('Decision reasons:')
for reason in selected.get('reasons', []):
    print('-', reason)

## 4. Persisted security events & verdict distributions

In [ ]:
raw_events = api_get('/v1/events', limit=500)
events_list = raw_events.get('events', raw_events) if isinstance(raw_events, dict) else raw_events
events = pd.DataFrame(events_list if isinstance(events_list, list) else [])
if events.empty or 'timestamp' not in events.columns or 'verdict' not in events.columns:
    print('No persisted events returned yet. Run queries or simulation scenarios first.')
else:
    events['timestamp'] = pd.to_datetime(events['timestamp'], errors='coerce', utc=True)
    display(events.head())
    fig = px.histogram(events, x='verdict', color='verdict',
                       category_orders={'verdict': ['ALLOW', 'FLAG', 'BLOCK']},
                       title='Persisted DNS security verdicts')
    fig.show()
    if 'domain_risk' in events.columns:
        fig = px.scatter(events, x='timestamp', y='domain_risk', color='verdict',
                         hover_data=[col for col in ['domain', 'client_ip', 'device_risk'] if col in events.columns],
                         title='Domain risk over time')
        fig.show()

## 5. Security trends and incidents

In [ ]:
trend_data = api_get('/v1/trends', hours=24)
trend = pd.DataFrame(trend_data.get('points', []) if isinstance(trend_data, dict) else [])
if not trend.empty and 'hour' in trend.columns:
    trend['hour'] = pd.to_datetime(trend['hour'], errors='coerce', utc=True)
    fig = go.Figure()
    if 'blocked_count' in trend.columns:
        fig.add_scatter(x=trend['hour'], y=trend['blocked_count'], mode='lines+markers', name='Blocked')
    if 'flagged_count' in trend.columns:
        fig.add_scatter(x=trend['hour'], y=trend['flagged_count'], mode='lines+markers', name='Flagged')
    if 'avg_domain_risk' in trend.columns:
        fig.add_scatter(x=trend['hour'], y=trend['avg_domain_risk'], mode='lines', name='Average domain risk', yaxis='y2')
    fig.update_layout(title='24-hour security trend', yaxis_title='Event count',
                      yaxis2=dict(title='Average risk', overlaying='y', side='right'))
    fig.show()
else:
    print('Hourly trend data unavailable or empty.')

incidents_raw = api_get('/v1/incidents')
incidents = incidents_raw if isinstance(incidents_raw, list) else []
if incidents:
    display(pd.DataFrame([{'id': item.get('id',''), 'device': item.get('device',''), 'severity': item.get('severity',''),
                           'timeline_entries': len(item.get('timeline', [])), 'summary': item.get('summary','')}
                          for item in incidents]))
else:
    print('No correlated incidents yet.')

## 6. Feed health & system statistics

In [ ]:
feed_health = api_get('/v1/feed-health')
model_monitoring = api_get('/v1/model-monitoring')
stats = api_get('/v1/stats')

print('Feed health')
display(pd.DataFrame(feed_health.get('feeds', [])))
print('Model monitoring')
display(pd.json_normalize(model_monitoring) if model_monitoring else pd.DataFrame())
print('Verdict summary')
display(pd.DataFrame(stats.get('by_verdict', [])))